In [1]:
from pyspark.sql import functions as F
from datetime import datetime

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
spark.sql("CREATE SCHEMA IF NOT EXISTS ops")

def log_run(step, ym, rows_in, rows_good, rows_rejected, status, message=""):
    cols = ["step","year_month","rows_in","rows_good","rows_rejected","status","message","_8"]
    row = [(step, ym, int(rows_in), int(rows_good), int(rows_rejected), status, message, datetime.now())]
    spark.createDataFrame(row, cols).write.mode("append").format("delta").saveAsTable("ops.pipeline_log")

def build_gold(name, df, rows_in):
    """Write one gold table and log the result. A failure is logged, then re-raised."""
    try:
        df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"gold.{name}")
        n = spark.table(f"gold.{name}").count()
        log_run(f"gold_{name}", "all", rows_in, n, rows_in - n, "SUCCESS")
        print(name, "OK", n)
    except Exception as e:
        log_run(f"gold_{name}", "all", rows_in, 0, 0, "FAILED", str(e)[:500])
        print(name, "FAILED:", str(e)[:200])
        raise

StatementMeta(, ca18dd4c-4c33-4c86-810d-2778795affef, 3, Finished, Available, Finished, False)

In [2]:
# dim_zone
zones = spark.table("silver.zones")
build_gold("dim_zone", zones.withColumnRenamed("location_id", "zone_key"), zones.count())

# dim_date
dates = spark.sql("SELECT explode(sequence(to_date('2024-01-01'), to_date('2024-12-31'), interval 1 day)) AS date")
dim_date = dates.select(
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"), "date",
    F.year("date").alias("year"), F.month("date").alias("month"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.dayofmonth("date").alias("day"),
    F.date_format("date", "EEEE").alias("day_name"),
    F.dayofweek("date").isin(1, 7).alias("is_weekend"))
build_gold("dim_date", dim_date, dates.count())

# dim_payment_type
pay = [(1,"Credit card"),(2,"Cash"),(3,"No charge"),(4,"Dispute"),(5,"Unknown"),(6,"Voided trip")]
dim_pay = spark.createDataFrame(pay, ["payment_type_key", "payment_type"])
build_gold("dim_payment_type", dim_pay, dim_pay.count())

# dim_weather
w = spark.table("silver.weather_hourly")
dim_weather = (w.withColumn("is_raining", F.col("precipitation_mm") > 0.1)
    .withColumn("temp_band_order", F.when(F.col("temp_c") < 0, 1).when(F.col("temp_c") < 10, 2).when(F.col("temp_c") < 20, 3).otherwise(4))
    .withColumn("temp_band", F.when(F.col("temp_c") < 0, "Freezing")
                              .when(F.col("temp_c") < 10, "Cold")
                              .when(F.col("temp_c") < 20, "Mild")
                              .otherwise("Warm")))
build_gold("dim_weather", dim_weather, w.count())

StatementMeta(, ca18dd4c-4c33-4c86-810d-2778795affef, 4, Finished, Available, Finished, False)

dim_zone OK 265
dim_date OK 366
dim_payment_type OK 6
dim_weather OK 2184


In [3]:
t = spark.table("silver.taxi_trips")
fact = t.select(
    "trip_id",
    F.date_format("pickup_ts", "yyyyMMdd").cast("int").alias("date_key"),
    "pickup_hours",
    F.col("pu_location_id").alias("pu_zone_key"),
    F.col("do_location_id").alias("do_zone_key"),
    F.col("payment_type").alias("payment_type_key"),
    "passenger_count", "trip_distance", "trip_duration_min",
    "fare_amount", "tip_amount", "total_amount", "tip_pct", "year_month")
build_gold("fact_trips", fact, t.count())


total = spark.table("gold.fact_trips").count()

# Orphan checks: fact rows whose keys don't exist in a dimension
f = spark.table("gold.fact_trips")
checks = {
    "orphan_pu_zone": """
        SELECT COUNT(*) FROM gold.fact_trips f
        LEFT JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
        WHERE z.zone_key IS NULL""",
    "orphan_do_zone": """
        SELECT COUNT(*) FROM gold.fact_trips f
        LEFT JOIN gold.dim_zone z ON f.do_zone_key = z.zone_key
        WHERE z.zone_key IS NULL""",
    "orphan_weather": """
        SELECT COUNT(*) FROM gold.fact_trips f
        LEFT JOIN gold.dim_weather w ON f.pickup_hours = w.weather_hour
        WHERE w.weather_hour IS NULL""",
    "orphan_date": """
        SELECT COUNT(*) FROM gold.fact_trips f
        LEFT JOIN gold.dim_date d ON f.date_key = d.date_key
        WHERE d.date_key IS NULL""",
    "orphan_payment": """
        SELECT COUNT(*) FROM gold.fact_trips f
        LEFT JOIN gold.dim_payment_type p ON f.payment_type_key = p.payment_type_key
        WHERE p.payment_type_key IS NULL""",
}

for name, q in checks.items():
    n = spark.sql(q).collect()[0][0]
    log_run(f"gold_check_{name}", "all", total, total - n, n, "SUCCESS" if n == 0 else "WARN")
    print(name, n)

StatementMeta(, ca18dd4c-4c33-4c86-810d-2778795affef, 5, Finished, Available, Finished, False)

fact_trips OK 0
orphan_pu_zone 0
orphan_do_zone 0
orphan_weather 0
orphan_date 0
orphan_payment 0


StatementMeta(, ca18dd4c-4c33-4c86-810d-2778795affef, 9, Finished, Available, Finished, False)

Error: [TABLE_OR_VIEW_NOT_FOUND] The table or view `bronze`.`taxi_trips` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 1 pos 14;
'Project [*]
+- 'UnresolvedRelation [bronze, taxi_trips], [], false
